In [1]:
%cd ~/Desktop/線上課程教材/pythan基礎觀念/爬蟲/好球帶研究室
%load_ext autoreload
%autoreload 2
import sys
print(sys.executable)

/Users/leochen/Desktop/線上課程教材/pythan基礎觀念/爬蟲/好球帶研究室
/Users/leochen/Desktop/線上課程教材/pythan基礎觀念/爬蟲/好球帶研究室/.venv/bin/python


In [2]:
%%writefile .streamlit/config.toml

[theme]
base = "dark"
primaryColor = "#F4B23E"
backgroundColor = "#0D1520"
secondaryBackgroundColor = "#142030"
textColor = "#E9EEF4"
font = "sans serif"

Writing .streamlit/config.toml


In [4]:
%%writefile app.py
import streamlit as st

st.set_page_config(page_title='好球帶研究室',page_icon='⚾',layout='wide')

pages = [
    st.Page("pages/home.py", title="首頁", default=True),
    st.Page("pages/academy.py", title="數據學院"),
    st.Page("pages/watch.py", title="球季觀察室"),
    st.Page("pages/predict.py", title="台灣大賽預測"),
]

st.navigation(pages,position='top').run()

Overwriting app.py


In [5]:
%%writefile src/ui.py

"""全站共用的樣式與小元件（卡片、數字方塊、標籤）。"""
import streamlit as st

COLORS = {"bg": "#0D1520", "surface": "#142030", "surface2": "#1B2A3D", "line": "#26384F",
          "fg": "#E9EEF4", "muted": "#8C9BB0", "accent": "#F4B23E",
          "good": "#46B98C", "bad": "#E8654F", "cold": "#4A86DE", "hot": "#E2483E"}

CSS = """
<style>
@import url('https://fonts.googleapis.com/css2?family=Barlow+Condensed:wght@600;700&family=Noto+Sans+TC:wght@400;700;900&display=swap');
html, body, [class*="css"] { font-family: 'Noto Sans TC', sans-serif; }
h1, h2, h3 { font-weight: 900 !important; letter-spacing: .02em; }
.num { font-family: 'Barlow Condensed', sans-serif; font-variant-numeric: tabular-nums; }
.eyebrow { color: #F4B23E; font-size: 13px; letter-spacing: .14em; font-weight: 700; margin: 0; }
.card { background: #142030; border: 1px solid #26384F; border-radius: 10px; padding: 18px 20px; margin-bottom: 12px; }
.kpi-label { color: #8C9BB0; font-size: 13px; }
.kpi-value { font-family: 'Barlow Condensed', sans-serif; font-size: 40px; font-weight: 700; line-height: 1.1; }
.kpi-who { font-weight: 700; }
.chip { display: inline-block; font-size: 12px; font-weight: 700; padding: 2px 9px;
        border-radius: 999px; border: 1px solid currentColor; margin-top: 4px; }
.chip.good { color: #46B98C; } .chip.bad { color: #E8654F; } .chip.warn { color: #F4B23E; }
.chip.cold { color: #4A86DE; } .chip.hot { color: #E2483E; }
.card ul { margin: 0; padding-left: 20px; } .card li { margin-bottom: 6px; }
</style>
"""

def setup(title):
    st.markdown(CSS, unsafe_allow_html=True)
    st.title(title)

def chip(text, tone="warn"):
    return f'<span class="chip {tone}">{text}</span>'

def kpi(label, value, who="", note=""):
    st.markdown(f'<div class="card"><div class="kpi-label">{label}</div>'
                f'<div class="kpi-value">{value}</div><div class="kpi-who">{who}</div>{note}</div>',
                unsafe_allow_html=True)

def bullet_card(title, items, color):
    lis = "".join(f"<li>{x}</li>" for x in items)
    st.markdown(f'<div class="card" style="border-top:4px solid {color}">'
                f'<b>{title}</b><ul style="margin-top:8px">{lis}</ul></div>', unsafe_allow_html=True)

def f3(x):
    return f"{x:.3f}".lstrip("0") if x >= 0 else "-" + f"{-x:.3f}".lstrip("0")

Writing src/ui.py


In [6]:
%%writefile src/data.py

"""網站讀資料的地方。只讀 data/clean 的 CSV，不在使用者開頁時爬網站。"""
from pathlib import Path

import pandas as pd
import streamlit as st

CLEAN = Path(__file__).resolve().parent.parent / "data" / "clean"

@st.cache_data
def load():
    teams = pd.read_csv(CLEAN / "teams.csv")
    batters = pd.read_csv(CLEAN / "batters.csv").merge(
        pd.read_csv(CLEAN / "batter_splits.csv"), left_on="player_id", right_on="batter_id", how="left")
    pitchers = pd.read_csv(CLEAN / "pitchers.csv").merge(
        pd.read_csv(CLEAN / "pitcher_splits.csv"), left_on="player_id", right_on="pitcher_id", how="left")
    batters["bats"] = batters["bats"].fillna("R")
    const = pd.read_csv(CLEAN / "league_constants.csv", index_col=0).iloc[:, 0]
    q = pitchers[pitchers["IP"] >= 30]
    lg = {"wOBA": (batters["wOBA_calc"] * batters["PA"]).sum() / batters["PA"].sum(),
          "BBp_p75": q["BBp"].quantile(0.75), "HR9_p75": q["HR9"].quantile(0.75)}
    return teams, batters, pitchers, const, lg

@st.cache_data
def load_pa():
    return pd.read_csv(CLEAN / "pa.csv", dtype={"result": str}, keep_default_na=False)

Writing src/data.py


In [7]:
%%writefile pages/home.py
import streamlit as st

from src import ui
from src.data import load

ui.setup("首頁")              # 每一頁換成自己的標題
teams, batters, pitchers, const, lg = load()
st.write(f"讀到 {len(batters)} 位打者、{len(pitchers)} 位投手。")
ui.kpi("測試卡片", ".319", "聯盟 wOBA")

Writing pages/home.py


In [8]:
%%writefile pages/academy.py
import streamlit as st
from src import ui

ui.setup("數據學院")

Writing pages/academy.py


In [9]:
%%writefile pages/watch.py
import streamlit as st
from src import ui

ui.setup("球季觀察室")

Writing pages/watch.py


In [10]:
%%writefile pages/predict.py
import streamlit as st
from src import ui

ui.setup("台灣大賽預測")

Writing pages/predict.py


In [11]:
!git add .
!git commit -m "Day 14: 網站骨架"

[main 6e5f5ed] Day 14: 網站骨架
 Committer: Leo Chen <leochen@LeodeMacBook-Air.local>
Your name and email address were configured automatically based
on your username and hostname. Please check that they are accurate.
You can suppress this message by setting them explicitly. Run the
following command and follow the instructions in your editor to edit
your configuration file:

    git config --global --edit

After doing this, you may fix the identity used for this commit with:

    git commit --amend --reset-author

 12 files changed, 590 insertions(+), 118 deletions(-)
 create mode 100644 .streamlit/config.toml
 create mode 100644 D09.ipynb
 create mode 100644 pages/academy.py
 create mode 100644 pages/home.py
 create mode 100644 pages/predict.py
 create mode 100644 pages/watch.py
 create mode 100644 src/data.py
 create mode 100644 src/ui.py


In [24]:
%%writefile pages/predict.py

import pandas as pd
import plotly.graph_objects as go
import streamlit as st

from src import config, fetch, model, rules, ui
from src.data import load

ui.setup('台灣大賽預測')
teams, batters, pitchers, const, lg=load()
A_name , B_name=config.TS_TEAMS
A=teams.set_index('team').loc[A_name]
B=teams.set_index('team').loc[B_name]

Overwriting pages/predict.py


In [33]:
%%writefile -a pages/predict.py

p_a,results,prob=model.simulate_series(A['pythag'],B['pythag'],config.TS_HOME_PATTERN,hfa=config.HOME_ADVANTAGE,need=config.TS_NEED)
st.markdown('<p class="eyebrow">系列賽預測 · 蒙地卡羅模擬 10,000 次</p>', unsafe_allow_html=True)

c1,c2=st.columns(2)

with c1:
    ui.kpi(f'{A_name}奪冠機率',f'{p_a:.0%}',f'畢氏勝率{ui.f3(A['pythag'])}')
with c2:
    ui.kpi(f'{B_name}奪冠機率',f'{1-p_a:.0%}',f'畢氏勝率{ui.f3(B['pythag'])}')

Appending to pages/predict.py


In [37]:
%%writefile -a pages/predict.py

order = ["4-0", "4-1", "4-2", "4-3", "3-4", "2-4", "1-4", "0-4"]
fig=go.Figure(go.Bar(
    x=order , y=[results.get(k,0)/100 for k in order],
    marker_color=[A["color"]] * 4 + [B["color"]] * 4,
    text=[f'{results.get(k,0)/100:.1f}%' for k in order],textposition="outside",cliponaxis=False))
fig.update_layout(title=f"系列賽結果分布（左邊 {A_name} 勝、右邊 {B_name} 勝）",
                  yaxis_title="機率（%）", height=320, margin=dict(t=50, b=20),
                  paper_bgcolor="rgba(0,0,0,0)", plot_bgcolor="rgba(0,0,0,0)")
st.plotly_chart(fig, width="stretch")

Appending to pages/predict.py


In [38]:
%%writefile -a pages/predict.py
with st.expander("這個機率怎麼來的？"):
    neutral = model.log5(A["pythag"], B["pythag"])
    st.markdown(f"""
1. **畢氏勝率**：用整季得分與失分估算實力。{A_name} 得 {A['RS']} 失 {A['RA']} → {ui.f3(A['pythag'])}；
   {B_name} 得 {B['RS']} 失 {B['RA']} → {ui.f3(B['pythag'])}。
2. **Log5**：換算成兩隊對戰的單場勝率。中立場地 {A_name} 勝率 {neutral:.1%}，
   主場再加 {config.HOME_ADVANTAGE:.1%}、客場減 {config.HOME_ADVANTAGE:.1%}。
3. **蒙地卡羅模擬**：照七戰四勝和主客場順序，把整個系列賽跑 10,000 次，數每隊贏了幾次。
""")
    st.latex(r"\text{畢氏勝率}=\frac{\text{得分}^{1.83}}{\text{得分}^{1.83}+\text{失分}^{1.83}}"
             r"\qquad P(A)=\frac{A-AB}{A+B-2AB}")

Appending to pages/predict.py


In [39]:
!git add .
!git commit -m "Day 15: 預測頁系列賽機率"

[main aeb553c] Day 15: 預測頁系列賽機率
 Committer: Leo Chen <leochen@LeodeMacBook-Air.local>
Your name and email address were configured automatically based
on your username and hostname. Please check that they are accurate.
You can suppress this message by setting them explicitly. Run the
following command and follow the instructions in your editor to edit
your configuration file:

    git config --global --edit

After doing this, you may fix the identity used for this commit with:

    git commit --amend --reset-author

 4 files changed, 467 insertions(+), 3 deletions(-)
 create mode 100644 .virtual_documents/D09.ipynb


In [40]:
%%writefile -a pages/predict.py

st.header('單場對戰分析')
g=st.selectbox('場次',range(len(config.TS_HOME_PATTERN)),
              format_func=lambda i : f'G{i+1} {A_name if config.TS_HOME_PATTERN[i] else B_name}主場')
a_home=config.TS_HOME_PATTERN[g]

Appending to pages/predict.py


In [58]:
%%writefile -a pages/predict.py
def team_picker(team,col):
    abbr=teams.set_index('team').loc[team,'abbr']
    sps=pitchers[(pitchers['abbr']==abbr) & (pitchers['role']=='先發')].sort_values('IP',ascending=False)
    bats = batters[batters["abbr"] == abbr].sort_values("PA", ascending=False)
    with col:
        st.subheader(team)
        sp_name = st.selectbox("先發投手", sps["name"],
                               format_func=lambda n: f"{n}（{rules.HAND.get(sps.set_index('name').loc[n, 'throws'], '')}・"
                                                     f"FIP {sps.set_index('name').loc[n, 'FIP_calc']:.2f}）",
                               key=f"sp_{abbr}")
        lineup_names = st.multiselect("先發打線（9 人）", bats["name"], default=list(bats["name"][:9]),
                                      max_selections=9, key=f"lu_{abbr}")
    sp = sps.set_index("name").loc[sp_name].copy()
    sp["name"] = sp_name
    lineup = bats[bats["name"].isin(lineup_names)]
    bench = bats[~bats["name"].isin(lineup_names)]
    return sp, lineup, bench, abbr



Appending to pages/predict.py


In [59]:
%%writefile -a pages/predict.py
ca, cb = st.columns(2)
sp_a, lu_a, bench_a, abbr_a = team_picker(A_name, ca)
sp_b, lu_b, bench_b, abbr_b = team_picker(B_name, cb)
if len(lu_a) != 9 or len(lu_b) != 9:
    st.info("兩隊都選滿 9 位打者後，會算出這場比賽的勝率。")
    st.stop()

Appending to pages/predict.py


In [60]:
%%writefile -a pages/predict.py
ctx = model.league_context(teams, pitchers, lg["wOBA"])
tp = pd.DataFrame(model.team_pitching(pitchers)).set_index("abbr")

def lineup_woba(lineup, sp):
    return lineup.apply(lambda b: rules.bat_woba(b, sp["throws"]), axis=1).mean()

inn = lambda sp: 5 if sp.get("wOBA_round3", 0) > lg["wOBA"] + 0.02 else 6
r_a = model.expected_runs(lineup_woba(lu_a, sp_b), lu_a["wOBA_calc"].mean(), sp_b["FIP_calc"],
                          tp.loc[abbr_b, "bullpen_fip"], tp.loc[abbr_b, "DER"], ctx, inn(sp_b), a_home)
r_b = model.expected_runs(lineup_woba(lu_b, sp_a), lu_b["wOBA_calc"].mean(), sp_a["FIP_calc"],
                          tp.loc[abbr_a, "bullpen_fip"], tp.loc[abbr_a, "DER"], ctx, inn(sp_a), not a_home)
p_game = model.win_prob_from_runs(r_a, r_b)

st.markdown(f'<div class="card"><p class="eyebrow">這場比賽的勝率</p>'
            f'<div class="kpi-value"><span style="color:{A["color"]}">{A_name} {p_game:.0%}</span> : '
            f'<span style="color:{B["color"]}">{1 - p_game:.0%} {B_name}</span></div>'
            f'<div class="kpi-label">預估得分 {r_a:.1f} : {r_b:.1f}</div></div>', unsafe_allow_html=True)

Appending to pages/predict.py


In [ ]:
!git add .
!git commit -m "Day 15: 預測頁系列賽機率"